In [1]:
!pip install transformers torch scikit-learn pandas

In [2]:
!pip install -U accelerate

   ---------------------------------------- 0.0/383.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/383.7 kB ? eta -:--:--
   - -------------------------------------- 10.2/383.7 kB ? eta -:--:--
   - -------------------------------------- 10.2/383.7 kB ? eta -:--:--
   - -------------------------------------- 10.2/383.7 kB ? eta -:--:--
   - -------------------------------------- 10.2/383.7 kB ? eta -:--:--
   - -------------------------------------- 10.2/383.7 kB ? eta -:--:--
   --- ----------------------------------- 30.7/383.7 kB 108.9 kB/s eta 0:00:04
   ---- ---------------------------------- 41.0/383.7 kB 109.3 kB/s eta 0:00:04
   ---- ---------------------------------- 41.0/383.7 kB 109.3 kB/s eta 0:00:04
   ------ -------------------------------- 61.4/383.7 kB 148.8 kB/s eta 0:00:03
   ------- ------------------------------- 71.7/383.7 kB 157.1 kB/s eta 0:00:02
   ----------- -------------------------- 112.6/383.7 kB 226.0 kB/s eta 0:00:02
   -------------- 

In [3]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments

C:\Users\KIIT0001\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\KIIT0001\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [4]:
df = pd.read_csv("safety_dataset.csv")

df.head()

,text,label
0,"""aaaand it's intercepted! [NAME] at the goal l...",0
1,I feel like ending my life,1
2,I'm not white.,0
3,Don't judge our love,0
4,I want to disappear forever,1


In [5]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42
)

In [6]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(train_texts, truncation=True, padding=True)
val_encodings = tokenizer(val_texts, truncation=True, padding=True)

In [7]:
len(train_encodings["input_ids"])

480

In [8]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [9]:
train_dataset = Dataset(train_encodings, train_labels)
val_dataset = Dataset(val_encodings, val_labels)

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
print(type(model))

<class 'transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification'>


In [12]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

Step,Training Loss
10,0.647270
20,0.538281
30,0.347680
40,0.196077
50,0.112365
60,0.045649
70,0.030470
80,0.022278
90,0.021788
100,0.016358


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=120, training_loss=0.16721612885594367, metrics={'train_runtime': 179.0472, 'train_samples_per_second': 5.362, 'train_steps_per_second': 0.67, 'total_flos': 9686678526720.0, 'train_loss': 0.16721612885594367, 'epoch': 2.0})

In [14]:
trainer.evaluate()

Training Loss,Validation Loss,Step
0.013906,0.011959,120


{'eval_loss': 0.011958643794059753}

In [15]:
model.save_pretrained("safety_model")
tokenizer.save_pretrained("safety_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('safety_model\\tokenizer_config.json', 'safety_model\\tokenizer.json')

In [16]:
from transformers import pipeline

classifier = pipeline("text-classification", model="safety_model", tokenizer="safety_model")

print(classifier("I feel like ending everything"))
print(classifier("I had a great day today"))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'LABEL_1', 'score': 0.9908689260482788}]
[{'label': 'LABEL_0', 'score': 0.9744511842727661}]
